In [1]:
%pip install mysql-connector-python pandas sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [6]:
import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Bsonureddy@123",
    database="cybersecurity_db",
    port=3306
)

if connection.is_connected():
    print("Connected successfully to MySQL!")

Connected successfully to MySQL!


In [7]:
cursor = connection.cursor()

cursor.execute("SHOW DATABASES")

for database in cursor:
    print(database[0])

cybersecurity_db
employee_details
information_schema
mysql
performance_schema
sakila
socialprachar_sql
sys
world


In [8]:
import pandas as pd
import mysql.connector
from pathlib import Path
import re

# 1. Folder containing your CSV files
folder_path = Path(r"C:\Users\pc\Downloads\Cybersecurity")

# 2. Connect to your database
cursor = connection.cursor()
cursor.execute("USE cybersecurity_db")

# 3. Find all CSV files
csv_files = sorted(
    list(folder_path.glob("*.csv"))
    + list(folder_path.glob("*.CSV"))
)

if not csv_files:
    print("No CSV files found. Check the folder path and file extensions.")
else:
    successful = []
    failed = []

    for file_path in csv_files:
        table_name = re.sub(
            r"[^a-zA-Z0-9_]+", "_", file_path.stem.lower()
        ).strip("_")

        try:
            # Read every column as text to preserve the original values
            try:
                df = pd.read_csv(
                    file_path,
                    encoding="utf-8-sig",
                    dtype=str,
                    keep_default_na=False
                )
            except UnicodeDecodeError:
                df = pd.read_csv(
                    file_path,
                    encoding="cp1252",
                    dtype=str,
                    keep_default_na=False
                )

            # Clean column names for MySQL
            cleaned_columns = []
            used_columns = set()

            for i, col in enumerate(df.columns):
                name = re.sub(
                    r"[^a-zA-Z0-9_]+", "_", str(col).strip().lower()
                ).strip("_")

                if not name:
                    name = f"column_{i + 1}"

                if name[0].isdigit():
                    name = f"col_{name}"

                original_name = name
                suffix = 2

                while name in used_columns:
                    name = f"{original_name}_{suffix}"
                    suffix += 1

                used_columns.add(name)
                cleaned_columns.append(name)

            df.columns = cleaned_columns

            # Create a table with text columns
            column_definitions = ", ".join(
                f"`{col}` LONGTEXT"
                for col in df.columns
            )

            cursor.execute(
                f"DROP TABLE IF EXISTS `{table_name}`"
            )

            cursor.execute(
                f"CREATE TABLE `{table_name}` "
                f"({column_definitions}) "
                "ENGINE=InnoDB DEFAULT CHARSET=utf8mb4"
            )

            # Insert rows in batches
            if len(df) > 0 and len(df.columns) > 0:
                columns_sql = ", ".join(
                    f"`{col}`" for col in df.columns
                )
                placeholders = ", ".join(
                    ["%s"] * len(df.columns)
                )

                insert_sql = (
                    f"INSERT INTO `{table_name}` "
                    f"({columns_sql}) VALUES ({placeholders})"
                )

                batch_size = 500

                for start in range(0, len(df), batch_size):
                    batch = df.iloc[start:start + batch_size]
                    values = [
                        tuple(row)
                        for row in batch.itertuples(
                            index=False, name=None
                        )
                    ]
                    cursor.executemany(insert_sql, values)

            connection.commit()

            successful.append((table_name, len(df), len(df.columns)))
            print(
                f"SUCCESS: {file_path.name} -> {table_name} "
                f"({len(df)} rows, {len(df.columns)} columns)"
            )

        except Exception as e:
            connection.rollback()
            failed.append((file_path.name, str(e)))
            print(f"FAILED: {file_path.name} -> {e}")

    cursor.close()

    print("\n========== IMPORT SUMMARY ==========")
    print("CSV files found:", len(csv_files))
    print("Successfully imported:", len(successful))
    print("Failed:", len(failed))

    print("\nImported tables:")
    for table_name, rows, cols in successful:
        print(f"{table_name}: {rows} rows, {cols} columns")

    if failed:
        print("\nFiles requiring attention:")
        for filename, error in failed:
            print(f"{filename}: {error}")

SUCCESS: associated detection strategies.csv -> associated_detection_strategies (697 rows, 12 columns)
SUCCESS: associated detection strategies.csv -> associated_detection_strategies (697 rows, 12 columns)
SUCCESS: associated mitigations.csv -> associated_mitigations (1448 rows, 13 columns)
SUCCESS: associated mitigations.csv -> associated_mitigations (1448 rows, 13 columns)
SUCCESS: citations.csv -> citations (319 rows, 3 columns)
SUCCESS: citations.csv -> citations (319 rows, 3 columns)
SUCCESS: citations_2.csv -> citations_2 (2257 rows, 3 columns)
SUCCESS: citations_2.csv -> citations_2 (2257 rows, 3 columns)
SUCCESS: citations_3.csv -> citations_3 (3998 rows, 3 columns)
SUCCESS: citations_3.csv -> citations_3 (3998 rows, 3 columns)
SUCCESS: Data Quality Report.csv -> data_quality_report (13 rows, 7 columns)
SUCCESS: Data Quality Report.csv -> data_quality_report (13 rows, 7 columns)
SUCCESS: defensive mappings.csv -> defensive_mappings (4160 rows, 11 columns)
SUCCESS: defensive map

In [9]:
table_name = "associated_detection_strategies"

cursor = connection.cursor()

try:
    cursor.execute("""
        ALTER TABLE `associated_detection_strategies`
            MODIFY `source_id` VARCHAR(20) NULL,
            MODIFY `source_name` VARCHAR(255) NULL,
            MODIFY `source_ref` VARCHAR(100) NULL,
            MODIFY `source_type` VARCHAR(50) NULL,
            MODIFY `mapping_type` VARCHAR(50) NULL,
            MODIFY `target_id` VARCHAR(20) NULL,
            MODIFY `target_name` VARCHAR(255) NULL,
            MODIFY `target_ref` VARCHAR(100) NULL,
            MODIFY `target_type` VARCHAR(50) NULL,
            MODIFY `stix_id` VARCHAR(100) NULL,
            MODIFY `created` DATE NULL,
            MODIFY `last_modified` DATE NULL
    """)

    connection.commit()
    print("Data types updated successfully.")

except Exception as e:
    connection.rollback()
    print("Update failed:", e)

finally:
    cursor.close()

Data types updated successfully.


In [10]:
cursor = connection.cursor()

try:
    # Get all tables in the database
    cursor.execute("SHOW TABLES")
    tables = [row[0] for row in cursor.fetchall()]

    print("Tables to be deleted:")
    for table in tables:
        print(table)

    confirm = input("\nType DELETE ALL TABLES to continue: ")

    if confirm == "DELETE ALL TABLES":
        cursor.execute("SET FOREIGN_KEY_CHECKS = 0")

        try:
            for table in tables:
                cursor.execute(f"DROP TABLE IF EXISTS `{table}`")

            connection.commit()
            print("\nAll tables deleted successfully!")

        finally:
            cursor.execute("SET FOREIGN_KEY_CHECKS = 1")

    else:
        print("\nDeletion cancelled. No tables were deleted.")

finally:
    cursor.close()

Tables to be deleted:
associated_detection_strategies
associated_mitigations
citations
citations_2
citations_3
data_quality_report
defensive_mappings
detectionstrategies
mitigations
procedure_examples
readme
relationships
tactics
techniques
techniques_addressed



Type DELETE ALL TABLES to continue:   DELETE ALL TABLES



Deletion cancelled. No tables were deleted.


In [11]:
cursor = connection.cursor()

try:
    # Get all tables in the database
    cursor.execute("SHOW TABLES")
    tables = [row[0] for row in cursor.fetchall()]

    print("Tables to be deleted:")
    for table in tables:
        print(table)

    confirm = input("\nType DELETE ALL TABLES to continue: ")

    if confirm == "DELETE ALL TABLES":
        cursor.execute("SET FOREIGN_KEY_CHECKS = 0")

        try:
            for table in tables:
                cursor.execute(f"DROP TABLE IF EXISTS `{table}`")

            connection.commit()
            print("\nAll tables deleted successfully!")

        finally:
            cursor.execute("SET FOREIGN_KEY_CHECKS = 1")

    else:
        print("\nDeletion cancelled. No tables were deleted.")

finally:
    cursor.close()

Tables to be deleted:
associated_detection_strategies
associated_mitigations
citations
citations_2
citations_3
data_quality_report
defensive_mappings
detectionstrategies
mitigations
procedure_examples
readme
relationships
tactics
techniques
techniques_addressed



Type DELETE ALL TABLES to continue:  DELETE ALL TABLES



All tables deleted successfully!


In [12]:
cursor = connection.cursor()

try:
    # Get all tables in the database
    cursor.execute("SHOW TABLES")
    tables = [row[0] for row in cursor.fetchall()]

    print("Tables to be deleted:")
    for table in tables:
        print(table)

    confirm = input("\nType DELETE ALL TABLES to continue: ")

    if confirm == "DELETE ALL TABLES":
        cursor.execute("SET FOREIGN_KEY_CHECKS = 0")

        try:
            for table in tables:
                cursor.execute(f"DROP TABLE IF EXISTS `{table}`")

            connection.commit()
            print("\nAll tables deleted successfully!")

        finally:
            cursor.execute("SET FOREIGN_KEY_CHECKS = 1")

    else:
        print("\nDeletion cancelled. No tables were deleted.")

finally:
    cursor.close()

Tables to be deleted:



Type DELETE ALL TABLES to continue:  DELETE ALL TABLES



All tables deleted successfully!


In [15]:
import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Bsonureddy@123",
    database="cybersecurity_db"
)

cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS test_table (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(100)
)
""")

connection.commit()

print("Table created successfully!")

Table created successfully!


In [20]:
import pandas as pd
import mysql.connector
from pathlib import Path
import re

# 1. Folder containing your CSV files
folder_path = Path(
    r"C:\Users\pc\Downloads\Cybersecurity"
)

# 2. Connect to your database
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Bsonureddy@123",
    database="cybersecurity_db"
)

cursor = connection.cursor()

# 3. Find all CSV files
csv_files = sorted(
    list(folder_path.glob("*.csv"))
    + list(folder_path.glob("*.CSV"))
)

if not csv_files:
    print("No CSV files found. Check the folder path and file extensions.")

else:
    successful = []
    failed = []

    for file_path in csv_files:

        # Convert filename into a valid MySQL table name
        table_name = re.sub(
            r"[^a-zA-Z0-9_]+",
            "_",
            file_path.stem.lower()
        ).strip("_")

        try:

            # 4. Read CSV
            try:
                df = pd.read_csv(
                    file_path,
                    encoding="utf-8-sig",
                    dtype=str,
                    keep_default_na=False
                )

            except UnicodeDecodeError:
                df = pd.read_csv(
                    file_path,
                    encoding="cp1252",
                    dtype=str,
                    keep_default_na=False
                )

            # 5. Clean column names
            cleaned_columns = []
            used_columns = set()

            for i, col in enumerate(df.columns):

                name = re.sub(
                    r"[^a-zA-Z0-9_]+",
                    "_",
                    str(col).strip().lower()
                ).strip("_")

                if not name:
                    name = f"column_{i + 1}"

                if name[0].isdigit():
                    name = f"col_{name}"

                original_name = name
                suffix = 2

                while name in used_columns:
                    name = f"{original_name}_{suffix}"
                    suffix += 1

                used_columns.add(name)
                cleaned_columns.append(name)

            df.columns = cleaned_columns

            # 6. Create table structure
            column_definitions = ", ".join(
                f"`{col}` LONGTEXT"
                for col in df.columns
            )

            # Delete existing table if it already exists
            cursor.execute(
                f"DROP TABLE IF EXISTS `{table_name}`"
            )

            # Create new table
            cursor.execute(
                f"""
                CREATE TABLE `{table_name}`
                ({column_definitions})
                ENGINE=InnoDB
                DEFAULT CHARSET=utf8mb4
                """
            )

            # 7. Insert data
            if len(df) > 0 and len(df.columns) > 0:

                columns_sql = ", ".join(
                    f"`{col}`"
                    for col in df.columns
                )

                placeholders = ", ".join(
                    ["%s"] * len(df.columns)
                )

                insert_sql = (
                    f"INSERT INTO `{table_name}` "
                    f"({columns_sql}) "
                    f"VALUES ({placeholders})"
                )

                # Insert in batches
                batch_size = 500

                for start in range(0, len(df), batch_size):

                    batch = df.iloc[
                        start:start + batch_size
                    ]

                    values = [
                        tuple(row)
                        for row in batch.itertuples(
                            index=False,
                            name=None
                        )
                    ]

                    cursor.executemany(
                        insert_sql,
                        values
                    )

            # 8. Save changes
            connection.commit()

            successful.append(
                (
                    table_name,
                    len(df),
                    len(df.columns)
                )
            )

            print(
                f"SUCCESS: {file_path.name} -> {table_name} "
                f"({len(df)} rows, {len(df.columns)} columns)"
            )

        except Exception as e:

            connection.rollback()

            failed.append(
                (
                    file_path.name,
                    str(e)
                )
            )

            print(
                f"FAILED: {file_path.name} -> {e}"
            )

    # 9. Close cursor
    cursor.close()

    # 10. Import summary
    print("\n========== IMPORT SUMMARY ==========")

    print(
        "CSV files found:",
        len(csv_files)
    )

    print(
        "Successfully imported:",
        len(successful)
    )

    print(
        "Failed:",
        len(failed)
    )

    # 11. Show imported tables
    print("\nImported tables:")

    for table_name, rows, cols in successful:

        print(
            f"{table_name}: "
            f"{rows} rows, "
            f"{cols} columns"
        )

    # 12. Show failed files
    if failed:

        print("\nFiles requiring attention:")

        for filename, error in failed:

            print(
                f"{filename}: {error}"
            )

print("\nDone!")

SUCCESS: associated detection strategies.csv -> associated_detection_strategies (699 rows, 12 columns)
SUCCESS: associated detection strategies.csv -> associated_detection_strategies (699 rows, 12 columns)
SUCCESS: associated mitigations.csv -> associated_mitigations (1450 rows, 13 columns)
SUCCESS: associated mitigations.csv -> associated_mitigations (1450 rows, 13 columns)
SUCCESS: citations.csv -> citations (321 rows, 3 columns)
SUCCESS: citations.csv -> citations (321 rows, 3 columns)
SUCCESS: citations_2.csv -> citations_2 (2259 rows, 3 columns)
SUCCESS: citations_2.csv -> citations_2 (2259 rows, 3 columns)
SUCCESS: citations_3.csv -> citations_3 (4000 rows, 3 columns)
SUCCESS: citations_3.csv -> citations_3 (4000 rows, 3 columns)
SUCCESS: Data Quality Report.csv -> data_quality_report (13 rows, 7 columns)
SUCCESS: Data Quality Report.csv -> data_quality_report (13 rows, 7 columns)
SUCCESS: defensive mappings.csv -> defensive_mappings (4162 rows, 11 columns)
SUCCESS: defensive map